# Selective Frequency Reconstruction

Standard MRI reconstruction applies IFFT to the full k-space, treating all frequencies equally.

This notebook shows what happens when you reconstruct from **only part of k-space** — isolating tissue types by their frequency signature, without any segmentation algorithm.

- **Low frequencies** (center of k-space) → coarse structure, bulk contrast, large anatomy
- **High frequencies** (edges of k-space) → fine detail, edges, boundaries
- **Band mask** → intermediate structures — specific tissue layers

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace, root_sum_of_squares
from kode.selective_reconstruct import (
    low_frequency_mask,
    high_frequency_mask,
    band_mask,
    selective_reconstruct,
)

In [ ]:
# Load a single slice from a fastMRI .h5 file
# Place your file in ../data/ and update the path below
H5_FILE = '../data/your_file.h5'
SLICE_IDX = 15

kspace = load_kspace(H5_FILE, slice_idx=SLICE_IDX)
print(f'K-space shape: {kspace.shape}  (coils, height, width)')

In [ ]:
_, H, W = kspace.shape

full_image    = root_sum_of_squares(kspace)
low_image     = selective_reconstruct(kspace, low_frequency_mask(H, W, fraction=0.08))
high_image    = selective_reconstruct(kspace, high_frequency_mask(H, W, fraction=0.08))
band_image    = selective_reconstruct(kspace, band_mask(H, W, inner=0.05, outer=0.25))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
titles = ['Full Reconstruction', 'Low Freq Only\n(coarse structure)', 'High Freq Only\n(edges + detail)', 'Band 5-25%\n(intermediate tissue)']
images = [full_image, low_image, high_image, band_image]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('Selective Frequency Reconstruction — Kode', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/selective_reconstruct.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize the masks themselves
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
masks = [
    low_frequency_mask(H, W, fraction=0.08),
    high_frequency_mask(H, W, fraction=0.08),
    band_mask(H, W, inner=0.05, outer=0.25),
]
mask_titles = ['Low Freq Mask', 'High Freq Mask', 'Band Mask']

for ax, mask, title in zip(axes, masks, mask_titles):
    ax.imshow(mask, cmap='viridis')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()